# 面试问题：怎样从零实现 MCP 风格 JSON-RPC 生命周期与能力协商？

可以直接复述的回答是：第一，客户端必须先 initialize，再发送 initialized 通知，之后才能 list 或 call 工具。第二，请求 id 在会话中唯一，通知不带 id。第三，双方只使用协商过的协议版本和能力。第四，方法参数要按 schema 校验，协议错误返回结构化 JSON-RPC error。第五，重复 id、越序调用和未知方法不能进入工具层。第六，用消息轨迹、状态迁移和错误码评估实现。下面构建一个离线 HR 工具服务器。

## 真实案例：员工助手连接 HR MCP 服务

消息序列包含 initialize、initialized、tools/list、两次 tools/call 和 ping，共 6 条。工具是年假余额查询与制度搜索，参数和返回均为脱敏教学数据。实现只覆盖解释生命周期所需的 JSON-RPC 子集，不代表完整 MCP 规范，也不建立网络连接。

In [1]:
messages = [  # 定义一条合法会话中的六个 JSON-RPC 消息
    {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-11-25", "capabilities": {"tools": True}}},  # 客户端先声明协议与工具能力
    {"jsonrpc": "2.0", "method": "notifications/initialized", "params": {}},  # 客户端确认初始化完成且通知不带 id
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}},  # 就绪后查询可用工具合同
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "leave_balance", "arguments": {"employee_id": "E-17"}}},  # 调用只读年假余额工具
    {"jsonrpc": "2.0", "id": 4, "method": "ping", "params": {}},  # 使用唯一请求 id 检查会话存活
    {"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "policy_search", "arguments": {"query": "年假结转"}}},  # 调用制度搜索工具
]  # 结束合法消息序列
print("协议输入：seq | id | method | params")  # 展示服务器实际接收的消息字段
for sequence, message in enumerate(messages, start=1):  # 逐条输出六个会话消息
    print(f"{sequence} | {message.get('id', '-')} | {message['method']} | {message['params']}")  # 呈现请求和通知的 id 差异


协议输入：seq | id | method | params
1 | 1 | initialize | {'protocolVersion': '2025-11-25', 'capabilities': {'tools': True}}
2 | - | notifications/initialized | {}
3 | 2 | tools/list | {}
4 | 3 | tools/call | {'name': 'leave_balance', 'arguments': {'employee_id': 'E-17'}}
5 | 4 | ping | {}
6 | 5 | tools/call | {'name': 'policy_search', 'arguments': {'query': '年假结转'}}


## Baseline / 基线：按 method 直接分发

最简单的 dispatcher 不维护生命周期，只要看到 tools/call 就执行。于是客户端可以在 initialize 前调用 HR 工具，重复 id 也不会被发现。

In [2]:
tool_results = {"leave_balance": {"days": 7.5, "expires": "2026-12-31"}, "policy_search": {"doc_id": "HR-22", "snippet": "年假最多结转 5 天"}}  # 定义两个离线 HR 工具结果
def baseline_dispatch(message):  # 实现不感知会话状态的方法分发基线
    if message["method"] == "tools/call":  # 只要方法名匹配就进入工具层
        return {"result": tool_results.get(message["params"]["name"])}  # 完全忽略初始化和能力协商
    return {"result": "accepted"}  # 其余方法统一返回接受
premature_call = {"jsonrpc": "2.0", "id": 99, "method": "tools/call", "params": {"name": "leave_balance", "arguments": {"employee_id": "E-17"}}}  # 构造初始化前的越序工具调用
baseline_premature = baseline_dispatch(premature_call)  # 用无状态 dispatcher 处理越序请求
print("初始化前 tools/call：", premature_call)  # 展示违反生命周期的原始消息
print("无状态基线返回：", baseline_premature)  # 展示工具被错误提前执行


初始化前 tools/call： {'jsonrpc': '2.0', 'id': 99, 'method': 'tools/call', 'params': {'name': 'leave_balance', 'arguments': {'employee_id': 'E-17'}}}
无状态基线返回： {'result': {'days': 7.5, 'expires': '2026-12-31'}}


## 核心实现：状态机、能力协商与请求 id 门禁

会话状态按 new → initializing → ready 迁移。每个请求先校验 JSON-RPC 版本和 id 唯一性，再检查当前状态是否允许该方法。

In [3]:
class ProtocolSession:  # 实现最小可观察 JSON-RPC 生命周期服务器
    def __init__(self):  # 初始化新连接的协议状态
        self.state = "new"  # 新连接尚未完成能力协商
        self.protocol_version = None  # 初始化前没有协商版本
        self.capabilities = set()  # 保存客户端与服务器能力交集
        self.seen_ids = set()  # 记录已使用请求 id 防止响应错配
        self.trace = []  # 保存方法、前后状态和结果类型
    def error(self, request_id, code, message):  # 构造规范化 JSON-RPC 错误对象
        return {"jsonrpc": "2.0", "id": request_id, "error": {"code": code, "message": message}}  # 返回不泄露内部栈的错误响应
    def handle(self, message):  # 按生命周期处理单条请求或通知
        before = self.state  # 保存处理前状态供轨迹审计
        request_id = message.get("id")  # 通知没有请求 id
        if message.get("jsonrpc") != "2.0":  # 只接受 JSON-RPC 2.0 信封
            response = self.error(request_id, -32600, "invalid_jsonrpc_version")  # 返回无效请求错误
        elif request_id is not None and request_id in self.seen_ids:  # 请求 id 在单会话中必须唯一
            response = self.error(request_id, -32600, "duplicate_request_id")  # 阻止响应关联歧义
        else:  # 信封和请求身份通过基础校验
            if request_id is not None:  # 只有请求需要登记 id
                self.seen_ids.add(request_id)  # 在方法处理前占用请求 id
            method = message["method"]  # 读取待分发的方法名称
            params = message.get("params", {})  # 缺省参数对象为空字典
            if method == "initialize" and self.state == "new":  # initialize 只允许在新会话调用一次
                requested_version = params.get("protocolVersion")  # 读取客户端期望协议版本
                if requested_version != "2025-11-25":  # 教学服务器只支持一个固定版本
                    response = self.error(request_id, -32602, "unsupported_protocol_version")  # 拒绝无法协商的协议版本
                else:  # 版本兼容时计算能力交集
                    self.protocol_version = requested_version  # 保存本次会话协商版本
                    self.capabilities = {name for name, enabled in params.get("capabilities", {}).items() if enabled and name == "tools"}  # 只协商服务器真实支持的 tools 能力
                    self.state = "initializing"  # 等待客户端 initialized 通知
                    response = {"jsonrpc": "2.0", "id": request_id, "result": {"protocolVersion": self.protocol_version, "capabilities": sorted(self.capabilities)}}  # 返回协商结果
            elif method == "notifications/initialized" and self.state == "initializing" and request_id is None:  # initialized 必须是无 id 通知
                self.state = "ready"  # 完成握手并开放工具方法
                response = None  # JSON-RPC 通知不产生响应
            elif method == "ping" and request_id is not None:  # ping 可检查已建立连接的存活
                response = {"jsonrpc": "2.0", "id": request_id, "result": {}}  # 返回空成功对象
            elif method in {"tools/list", "tools/call"} and self.state != "ready":  # 工具方法只能在握手完成后调用
                response = self.error(request_id, -32002, "session_not_ready")  # 返回生命周期错误且不进入工具层
            elif method == "tools/list" and "tools" in self.capabilities:  # 只向协商过 tools 能力的客户端列目录
                tools = [{"name": "leave_balance", "required": ["employee_id"]}, {"name": "policy_search", "required": ["query"]}]  # 定义两个工具的最小 schema
                response = {"jsonrpc": "2.0", "id": request_id, "result": {"tools": tools}}  # 返回可调用工具合同
            elif method == "tools/call" and "tools" in self.capabilities:  # 工具调用要求能力已协商
                name = params.get("name")  # 读取客户端选择的工具名称
                required = {"leave_balance": "employee_id", "policy_search": "query"}.get(name)  # 获取当前工具必需参数
                if required is None or required not in params.get("arguments", {}):  # 未知工具或缺参数不能进入执行器
                    response = self.error(request_id, -32602, "invalid_tool_arguments")  # 返回参数合同错误
                else:  # schema 通过后读取离线工具结果
                    response = {"jsonrpc": "2.0", "id": request_id, "result": tool_results[name]}  # 返回确定性 HR 结果
            else:  # 未知方法或未协商能力统一拒绝
                response = self.error(request_id, -32601, "method_not_available")  # 返回标准方法不可用错误
        outcome = "notification" if response is None else ("error" if "error" in response else "result")  # 归一化当前处理结果类型
        self.trace.append((message["method"], before, self.state, outcome))  # 写入完整状态迁移轨迹
        return response  # 返回响应或通知空值
session = ProtocolSession()  # 创建一条全新的协议会话
responses = [session.handle(message) for message in messages]  # 按顺序处理六条合法消息
print("生命周期轨迹：method | before | after | outcome")  # 输出状态机最关键的中间过程
for event in session.trace:  # 逐条展示方法和状态迁移
    print(" | ".join(event))  # 格式化协议轨迹供人工检查


生命周期轨迹：method | before | after | outcome
initialize | new | initializing | result
notifications/initialized | initializing | ready | notification
tools/list | ready | ready | result
tools/call | ready | ready | result
ping | ready | ready | result
tools/call | ready | ready | result


## 失败案例与修正：越序调用、重复 id 和版本不兼容

三个错误分别对应生命周期、响应关联和版本协商。安全实现返回结构化错误，并证明工具层没有在错误状态下执行。

In [4]:
premature_session = ProtocolSession()  # 创建尚未初始化的新会话
safe_premature = premature_session.handle(premature_call)  # 尝试在 new 状态调用工具
duplicate_message = {"jsonrpc": "2.0", "id": 2, "method": "ping", "params": {}}  # 复用合法会话中已经使用的请求 id
duplicate_response = session.handle(duplicate_message)  # 在已就绪会话中触发 id 门禁
version_session = ProtocolSession()  # 创建用于版本冲突测试的独立会话
bad_version = {"jsonrpc": "2.0", "id": 7, "method": "initialize", "params": {"protocolVersion": "2024-01-01", "capabilities": {"tools": True}}}  # 构造服务器不支持的旧协议版本
version_response = version_session.handle(bad_version)  # 尝试协商不兼容版本
print("越序调用错误：", safe_premature)  # 展示 session_not_ready 结构化响应
print("重复 id 错误：", duplicate_response)  # 展示 duplicate_request_id 响应
print("版本协商错误：", version_response)  # 展示 unsupported_protocol_version 响应


越序调用错误： {'jsonrpc': '2.0', 'id': 99, 'error': {'code': -32002, 'message': 'session_not_ready'}}
重复 id 错误： {'jsonrpc': '2.0', 'id': 2, 'error': {'code': -32600, 'message': 'duplicate_request_id'}}
版本协商错误： {'jsonrpc': '2.0', 'id': 7, 'error': {'code': -32602, 'message': 'unsupported_protocol_version'}}


## 结果表：合法消息与错误消息处理统计

In [5]:
successful_responses = sum(response is not None and "result" in response for response in responses)  # 统计合法序列中的成功请求响应
notifications = sum(response is None for response in responses)  # 统计不应返回响应的 initialized 通知
failure_responses = [safe_premature, duplicate_response, version_response]  # 汇总三类失败案例响应
error_codes = [response["error"]["code"] for response in failure_responses]  # 提取可监控 JSON-RPC 错误码
print("合法序列响应：id | outcome | payload")  # 输出每条合法请求的结果摘要
for message, response in zip(messages, responses):  # 将请求与对应响应按顺序关联
    outcome = "notification" if response is None else ("error" if "error" in response else "result")  # 计算当前消息的响应类型
    print(f"{message.get('id', '-')} | {outcome} | {response}")  # 展示通知无响应和请求有响应的区别
print(f"汇总：final_state={session.state}，success={successful_responses}，notifications={notifications}，failure_codes={error_codes}")  # 输出生命周期与错误处理指标


合法序列响应：id | outcome | payload
1 | result | {'jsonrpc': '2.0', 'id': 1, 'result': {'protocolVersion': '2025-11-25', 'capabilities': ['tools']}}
- | notification | None
2 | result | {'jsonrpc': '2.0', 'id': 2, 'result': {'tools': [{'name': 'leave_balance', 'required': ['employee_id']}, {'name': 'policy_search', 'required': ['query']}]}}
3 | result | {'jsonrpc': '2.0', 'id': 3, 'result': {'days': 7.5, 'expires': '2026-12-31'}}
4 | result | {'jsonrpc': '2.0', 'id': 4, 'result': {}}
5 | result | {'jsonrpc': '2.0', 'id': 5, 'result': {'doc_id': 'HR-22', 'snippet': '年假最多结转 5 天'}}
汇总：final_state=ready，success=5，notifications=1，failure_codes=[-32002, -32600, -32602]


## 结果解读

合法轨迹严格经历 new → initializing → ready，initialized 通知没有响应，之后两个工具调用返回可读 HR 结果。初始化前调用返回 -32002，重复 id 返回 -32600，不兼容版本返回 -32602。状态机的价值是让错误在进入工具执行层之前失败，而不是等副作用发生后补救。

## 生产边界

完整实现还需内容长度限制、批量请求、取消、进度通知、超时、OAuth 资源受众、工具 schema 和传输层断连恢复。协议版本与方法集合应依据官方规范测试，本例的版本字符串和子集仅供教学。网络服务还必须防止跨连接复用 request id 状态，并对工具结果做数据分级。

## 最小回归测试

In [6]:
assert len(messages) >= 5  # 保证案例覆盖完整握手和多个工具请求
assert session.state == "ready"  # 保证合法序列最终进入可调用工具的就绪状态
assert responses[1] is None  # 保证 initialized 通知不生成 JSON-RPC 响应
assert responses[3]["result"]["days"] == 7.5  # 保证合法年假工具调用返回确定性结果
assert safe_premature["error"]["message"] == "session_not_ready"  # 保证越序调用在工具层前失败
assert duplicate_response["error"]["message"] == "duplicate_request_id"  # 保证会话请求 id 不可复用
assert version_response["error"]["message"] == "unsupported_protocol_version"  # 保证不兼容协议版本不会进入初始化状态
